In [ ]:
cell_str = r'''
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// Configuration
// - BM, BN: size of the tile, work for each Thread Block
// - BK: step size, how many elements are loaded together for the multiplication
// - TM, TN: size of the block computed by each single thread
#define BM 128
#define BN 128
#define BK 16
#define TM 8
#define TN 8

// Dynamic calculation of threads per block based on the tile dimensions
#define NUM_THREADS ((BN / TN) * (BM / TM))


// ---- KERNEL CUDA: 2D Register Tiling ----
__global__ void MatMult(int n, const float *A, const float *B, float *C) {

    // Shared Memory exploited by the Thread Block for the computation of the matrix slice
    __shared__ float sA[BM][BK];
    __shared__ float sB[BK][BN];

    // Local coordinates
    int bx = blockIdx.x;
    int by = blockIdx.y;
    int tx = threadIdx.x;
    int ty = threadIdx.y;

    // Registers to store the partial results of the multiplication, in order to maximize the speed
    float res[TM][TN] = {0.0f};
    float a_reg[TM];
    float b_reg[TN];

    // Linear index of the thread for Decoupled Loading
    int tid = ty * (BN / TN) + tx;

    // Loop: slicing along the k dimension (BK=16)
    for (int p = 0; p < (n + BK - 1) / BK; ++p) {

        // Decoupled Loading, each thread transfers contiguous elements from the Global Memory to the Shared Memory Tile A and B
        // - Linear index idx (Memory Coalescing)
        // - a_row, a_col: 2D coordinates within the Shared Memory
        // - g_a_row, g_a_col: transformation in global coordinates
        // - Saving in Shared Memory sA of the value or padding (0) if out of the matrix borders

        // Loading of a slice of the Tile A
        for (int i = 0; i < (BM * BK) / NUM_THREADS; ++i) {
            int idx = i * NUM_THREADS + tid;
            int a_row = idx / BK;
            int a_col = idx % BK;
            int g_a_row = by * BM + a_row;
            int g_a_col = p * BK + a_col;
            sA[a_row][a_col] = (g_a_row < n && g_a_col < n) ? A[g_a_row * n + g_a_col] : 0.0f;
        }

        // Loading of a slice of the Tile B
        for (int i = 0; i < (BK * BN) / NUM_THREADS; ++i) {
            int idx = i * NUM_THREADS + tid;
            int b_row = idx / BN;
            int b_col = idx % BN;
            int g_b_row = p * BK + b_row;
            int g_b_col = bx * BN + b_col;
            sB[b_row][b_col] = (g_b_row < n && g_b_col < n) ? B[g_b_row * n + g_b_col] : 0.0f;
        }

        // Synchronization Barrier: ensures that all threads have finished loading the data before to start the computation
        __syncthreads();


        // Register Promotion
        // - Moving data from Shared Memory to registers for the computation
        #pragma unroll
        for (int k = 0; k < BK; ++k) {
            for(int i=0; i<TM; i++) a_reg[i] = sA[ty * TM + i][k];
            for(int j=0; j<TN; j++) b_reg[j] = sB[k][tx * TN + j];

            for(int i=0; i<TM; i++) {
                for(int j=0; j<TN; j++) {
                    res[i][j] += a_reg[i] * b_reg[j];
                }
            }
        }
        __syncthreads();
    }


    // Saving in the Global Memory C
    // - Each thread saves its 64 results stored in the Registers to the Global Memory
    int row = by * BM + ty * TM;
    int col = bx * BN + tx * TN;

    for (int i = 0; i < TM; ++i) {
        for (int j = 0; j < TN; ++j) {
            int c_row = row + i;
            int c_col = col + j;
            if (c_row < n && c_col < n) {
                C[c_row * n + c_col] = res[i][j];
            }
        }
    }
}


// ---- HOST ----
int main(int argc, char **argv) {

    if (argc < 2) {
        fprintf(stderr, "Error: missing argument in %s \n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);

    // Host Memory Allocation (1D Flattening)
    size_t bytes = n * n * sizeof(float);
    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    // Initialization
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    // Device Memory Allocation
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Copy from Host to Device
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Execution configuration for the Kernel launch: Grid and Block dimensions
    dim3 threadsPerBlock(BN / TN, BM / TM);
    dim3 blocksPerGrid((n + BN - 1) / BN, (n + BM - 1) / BM);

    printf("Starting the computation...\n");

    // CUDA Event for Timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Kernel Launch + Time Measurement
    cudaEventRecord(start);
    MatMult<<<blocksPerGrid, threadsPerBlock>>>(n, d_a, d_b, d_c);
    cudaEventRecord(stop);

    // Synchronization: Host waits for Device execution
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Fast Execution Time: %.4f seconds\n", milliseconds / 1000.0f);

    // Copy from Device to Host
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // Results
    FILE *f = fopen("mat-res-fast.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int limit = (n < 1000) ? n : 1000;
        for (int i = 0; i < limit; i++) {
            for (int j = 0; j < limit; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    // Memory deallocation
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_matrixmult_opt.cu', 'w') as f:
    f.write(cell_str)

In [80]:
!nvcc -arch=sm_75 -O3 cuda_matrixmult_opt.cu -o cuda_matrixmult_opt

In [81]:
!nvprof ./cuda_matrixmult_opt 20000

==18614== NVPROF is profiling process 18614, command: ./cuda_matrixmult_opt 20000
Starting the computation...
Fast Execution Time: 3.9749 seconds
==18614== Profiling application: ./cuda_matrixmult_opt 20000
==18614== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   78.54%  3.97455s         1  3.97455s  3.97455s  3.97455s  MatMult(int, float const *, float const *, float*)
                   14.33%  725.12ms         2  362.56ms  354.00ms  371.12ms  [CUDA memcpy HtoD]
                    7.13%  360.73ms         1  360.73ms  360.73ms  360.73ms  [CUDA memcpy DtoH]
      API calls:   75.66%  3.97456s         1  3.97456s  3.97456s  3.97456s  cudaEventSynchronize
                   20.69%  1.08682s         3  362.27ms  354.22ms  371.42ms  cudaMemcpy
                    3.48%  182.91ms         3  60.970ms  144.41us  182.62ms  cudaMalloc
                    0.11%  5.9760ms         3  1.9920ms  1.1235ms  2.7773ms  cudaFree
   